# 03. Train one run (R00 to R03)

Each run is `configs/base.yaml` plus one override file, and changes exactly one thing:

| Run | Config | What changes | Dataset |
|---|---|---|---|
| R00-base | `configs/R00-base.yaml` | nothing | plain (`/content/yolo`) |
| R01-mixup | `configs/R01-mixup.yaml` | mixup 0.3 | plain |
| R02-rfs | `configs/R02-rfs.yaml` | repeat factor sampling, t 0.3 (train images only) | oversampled (`/content/yolo_rfs`) |
| R03-clspw | `configs/R03-clspw.yaml` | cls_pw 0.5 (class-weighted loss) | plain |

Set `RUN` below, then run all cells. Use a GPU runtime (Runtime > Change runtime type > T4 GPU).
Before starting, add the run to the Run tracker in the shared Google Doc.

## Parameters

In [ ]:
RUN = "R01-mixup"            # R00-base, R01-mixup, R02-rfs or R03-clspw

# PLACEHOLDERS: None keeps the config value (yolo11s.pt, 40 epochs, batch 16).
# Set them to match Neo's baseline so every run uses the same numbers.
MODEL  = None                # e.g. "yolo11s.pt"
EPOCHS = None                # e.g. 30
BATCH  = None                # e.g. 8 after an out-of-memory error (note it in the tracker)

NDJSON       = "../data/kara-agro-ai-maize.ndjson"
SPLIT_CSV    = "../splits/split_v1.csv"
DRIVE_IMAGES = "/content/drive/Shared drives/<team drive>/kara-agro/images_1280"   # same as 02-split
RUNS_DIR     = "/content/drive/Shared drives/<team drive>/kara-agro/runs"          # weights survive the session
YOLO_DIR     = "/content/yolo"        # fast local disk, rebuilt each session
RFS_DIR      = "/content/yolo_rfs"    # oversampled copy for R02 (symlinks, so it is small)
SEED = 42

In [ ]:
# Colab setup: Drive, and the Ultralytics version Neo used (latest unless told otherwise).
from google.colab import drive
drive.mount("/content/drive")
!pip install -q -U ultralytics
import ultralytics, sys
sys.path.append("..")
print("ultralytics", ultralytics.__version__, "  <- write this in the run tracker")

## 1. Dataset on local disk

Builds `/content/yolo` from the split CSV and the Drive images if it is not there yet (same as step 3 of `02-split.ipynb`).
For R02 it also builds the oversampled copy with `src/oversample.py`: every train image is repeated r times, r = max over its classes of max(1, sqrt(0.3 / f_c)), where f_c is the share of train images containing class c (LVIS repeat factor sampling, Gupta et al. 2019). Expect egg images about 4 times, larva about 2 times, about 13,400 train images in total. Val and test are not changed.

In [ ]:
import os
from src import make_yolo_dataset
if not os.path.exists(f"{YOLO_DIR}/dataset.yaml"):
    make_yolo_dataset.run(SPLIT_CSV, DRIVE_IMAGES, NDJSON, YOLO_DIR, neg_fraction=1.0, seed=SEED)
DATA = f"{YOLO_DIR}/dataset.yaml"

if RUN == "R02-rfs":
    from src import oversample
    if not os.path.exists(f"{RFS_DIR}/dataset.yaml"):
        oversample.build(YOLO_DIR, RFS_DIR, t=0.3, seed=SEED)
    DATA = f"{RFS_DIR}/dataset.yaml"
print("training data:", DATA)

## 2. Check the settings (no training)

In [ ]:
extra = []
if MODEL:  extra += ["--model", MODEL]
if EPOCHS: extra += ["--epochs", str(EPOCHS)]
if BATCH:  extra += ["--batch", str(BATCH)]
ARGS = [f"configs/{RUN}.yaml", "--data", DATA, "--project", RUNS_DIR, *extra]
ARGS_STR = " ".join(f'"{a}"' for a in ARGS)
!cd .. && python -m src.train {ARGS_STR} --dry-run

## 3. Train

Optional first: a smoke test with 1 epoch on 10% of the data, in a throwaway folder:
`!cd .. && python -m src.train {ARGS_STR} --name smoke-{RUN} --epochs 1 --set fraction=0.1`

For R03, copy the `Class weights: [...]` line from the log into the run tracker notes (it is also saved in `run_info.json`).

In [ ]:
!cd .. && python -m src.train {ARGS_STR}

## If Colab disconnected: resume from the last checkpoint

In [ ]:
# !cd .. && python -m src.train {ARGS_STR} --resume

## 4. Next

Evaluate with `04-evaluate.ipynb` (always on the plain dataset, also for R02), copy `args.yaml`, `run_config.yaml` and `run_info.json` into `results/<run>/` on the run's `exp/` branch, and fill in the run tracker.